# 🌍 Company Universe - Consolidation Complète

Ce notebook génère le **Company Universe consolidé** en fusionnant toutes les sources de données disponibles.

**Sources de données :**
- ✅ **Key Market Data** : Données de marché (S&P 500 composition + performance)
- ✅ **Data Points 10-K** : Extraction géographie, segments, supply chain (~97 entreprises)
- ⚠️ **Legal Data** : Extraction depuis documents réglementaires (à venir)

**Objectif :**
- Fusionner toutes les sources par ticker
- Créer une structure unifiée pour chaque entreprise
- Préparer la base de données pour l'analyse d'impact réglementaire


## Étape 1 : Imports et Configuration


In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import Dict, Optional, List
from collections import defaultdict
import os
from tqdm import tqdm

# Configuration des chemins
BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data' / 'generated'

KEY_MARKET_DATA_FILE = DATA_DIR / 'key_market_data' / 'all_market_data.json'
DATA_POINTS_DIR = DATA_DIR / 'extracted_data_points'
OUTPUT_DIR = DATA_DIR / 'company_universe'

# Créer le dossier de sortie
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier de sortie : {OUTPUT_DIR}")


## Étape 2 : Charger Key Market Data


In [ ]:
print("📊 Chargement des Key Market Data...")

with open(KEY_MARKET_DATA_FILE, 'r', encoding='utf-8') as f:
    key_market_data = json.load(f)

print(f"✅ {len(key_market_data)} entreprises dans Key Market Data")

# Afficher un exemple
example_ticker = 'AAPL'
if example_ticker in key_market_data:
    print(f"\n📋 Exemple ({example_ticker}):")
    print(json.dumps(key_market_data[example_ticker], indent=2, ensure_ascii=False))


## Étape 3 : Charger Data Points 10-K
 

In [ ]:
print("📄 Chargement des Data Points 10-K...")

# Lister tous les fichiers data points
data_points_files = list(DATA_POINTS_DIR.glob('*_data_points.json'))
print(f"📁 {len(data_points_files)} fichiers trouvés")

# Charger tous les data points
data_points = {}
for file_path in tqdm(data_points_files, desc="Chargement"):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            ticker = data.get('ticker', '').upper()
            if ticker:
                data_points[ticker] = data
    except Exception as e:
        print(f"⚠️ Erreur lors du chargement de {file_path.name}: {e}")

print(f"\n✅ {len(data_points)} entreprises avec Data Points 10-K")

# Afficher un exemple
if example_ticker in data_points:
    print(f"\n📋 Exemple ({example_ticker}):")
    print(f"   - Type: {data_points[example_ticker].get('company_type', 'N/A')}")
    print(f"   - Segments: {len(data_points[example_ticker].get('segments', []))} segments")
    print(f"   - Pays: {len(data_points[example_ticker].get('geography', {}).get('countries', []))} pays")


## Étape 4 : Charger Legal Data


In [ ]:
# TODO: Charger les Legal Data quand disponibles
# Structure attendue: {ticker: [list of regulations]}

legal_data_dir = DATA_DIR / 'legal_data'
legal_data = {}

if legal_data_dir.exists():
    print("⚖️ Chargement des Legal Data...")
    legal_files = list(legal_data_dir.glob('*.json'))
    
    for file_path in legal_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Structure à adapter selon le format réel
                # Exemple: {ticker: regulations}
                if isinstance(data, dict):
                    legal_data.update(data)
        except Exception as e:
            print(f"⚠️ Erreur lors du chargement de {file_path.name}: {e}")
    
    print(f"✅ {len(legal_data)} entreprises avec Legal Data")
else:
    print("⚠️ Legal Data non disponible (dossier non trouvé)")
    print(f"   Chemin attendu: {legal_data_dir}")


## Étape 5 : Fusionner les Données par Entreprise


In [ ]:
def merge_company_data(
    ticker: str,
    market_data: Optional[Dict],
    data_points: Optional[Dict],
    legal_data: Optional[Dict]
) -> Dict:
    """
    Fusionne toutes les sources de données pour une entreprise
    """
    company = {
        'ticker': ticker,
        'data_sources': {
            'has_market_data': market_data is not None,
            'has_data_points': data_points is not None,
            'has_legal_data': legal_data is not None
        },
        'market_data': market_data or {},
        'data_points': data_points or {},
        'legal_data': legal_data or {}
    }
    
    # Extraire le nom de l'entreprise depuis n'importe quelle source
    company_name = (
        market_data.get('company_name') if market_data
        else data_points.get('company_name') if data_points
        else None
    )
    
    if company_name:
        company['company_name'] = company_name
    
    return company

# Obtenir tous les tickers uniques
all_tickers = set(key_market_data.keys())
all_tickers.update(data_points.keys())
if legal_data:
    all_tickers.update(legal_data.keys())

print(f"📊 Total de {len(all_tickers)} entreprises uniques à traiter")


## Étape 6 : Générer le Company Universe


In [ ]:
print("🌍 Génération du Company Universe...")

company_universe = {}
stats = {
    'total_companies': len(all_tickers),
    'with_market_data': 0,
    'with_data_points': 0,
    'with_legal_data': 0,
    'complete_data': 0  # Toutes les sources disponibles
}

for ticker in tqdm(sorted(all_tickers), desc="Fusion"):
    market_data = key_market_data.get(ticker)
    dp_data = data_points.get(ticker)
    legal = legal_data.get(ticker) if legal_data else None
    
    # Mettre à jour les statistiques
    if market_data:
        stats['with_market_data'] += 1
    if dp_data:
        stats['with_data_points'] += 1
    if legal:
        stats['with_legal_data'] += 1
    if market_data and dp_data and legal:
        stats['complete_data'] += 1
    
    # Fusionner les données
    company_universe[ticker] = merge_company_data(ticker, market_data, dp_data, legal)

print(f"\n✅ Company Universe généré avec {len(company_universe)} entreprises")
print(f"\n📊 Statistiques:")
print(f"   - Avec Market Data: {stats['with_market_data']}")
print(f"   - Avec Data Points 10-K: {stats['with_data_points']}")
print(f"   - Avec Legal Data: {stats['with_legal_data']}")
print(f"   - Données complètes (3 sources): {stats['complete_data']}")


## Étape 7 : Sauvegarder le Company Universe


In [ ]:
# Sauvegarder en JSON complet (AVANT enrichissement, sera mis à jour après si enrichissement activé)
output_file_json = OUTPUT_DIR / 'company_universe.json'
with open(output_file_json, 'w', encoding='utf-8') as f:
    json.dump(company_universe, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier JSON sauvegardé: {output_file_json}")
if output_file_json.exists():
    print(f"   Taille: {output_file_json.stat().st_size / 1024 / 1024:.2f} MB")
    print("💡 Note: Si enrichissement activé, ce fichier sera mis à jour après")

# Sauvegarder aussi un résumé en CSV pour analyse rapide
summary_rows = []
for ticker, company in company_universe.items():
    row = {
        'ticker': ticker,
        'company_name': company.get('company_name', ''),
        'has_market_data': company['data_sources']['has_market_data'],
        'has_data_points': company['data_sources']['has_data_points'],
        'has_legal_data': company['data_sources']['has_legal_data']
    }
    
    # Ajouter quelques métriques clés si disponibles
    market = company.get('market_data', {})
    if market:
        row['market_cap'] = market.get('market_cap')
        row['sp500_weight'] = market.get('sp500_weight')
        row['pe_ratio'] = market.get('pe_ratio')
    
    dp = company.get('data_points', {})
    if dp:
        row['company_type'] = dp.get('company_type', '')
        geo = dp.get('geography', {})
        row['num_countries'] = len(geo.get('countries', []))
        row['has_eu'] = geo.get('has_eu', False)
        row['has_na'] = geo.get('has_na', False)
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
output_file_csv = OUTPUT_DIR / 'company_universe_summary.csv'
summary_df.to_csv(output_file_csv, index=False, encoding='utf-8')

print(f"✅ Fichier CSV résumé sauvegardé: {output_file_csv}")
print(f"\n📋 Aperçu du résumé:")
print(summary_df.head(10).to_string())


## Étape 8 : Enrichissement avec Yahoo Finance (Optionnel)

Enrichissement avec données temps réel pour améliorer précision des analyses d'impact.


In [ ]:
# Importer la fonction d'enrichissement
import sys
sys.path.append(str(BASE_DIR / 'scripts'))
try:
    from enrich_with_yfinance import enrich_company_universe
    
    # Option : Enrichir seulement les top N entreprises (pour limiter appels API)
    # Mettre None pour désactiver l'enrichissement
    ENRICH_TOP_N = 50  # Changer ce nombre selon besoin (None = désactiver)
    
    if ENRICH_TOP_N:
        print(f"📊 Enrichissement avec Yahoo Finance des {ENRICH_TOP_N} premières entreprises...")
        print("⏱️  Cela peut prendre quelques minutes...")
        
        company_universe_enriched = enrich_company_universe(
            company_universe, 
            limit_top_n=ENRICH_TOP_N
        )
        
        # Remplacer le Company Universe avec version enrichie
        company_universe = company_universe_enriched
        
        print("✅ Enrichissement terminé!")
        print("\n💡 Les données enrichies sont dans: market_data.realtime")
        print("   - current_price: Prix actuel (vs csv_price du CSV)")
        print("   - beta: Volatilité (pour ajuster risque)")
        print("   - volatility_factor: Facteur d'ajustement du risque")
        
        # Afficher un exemple
        enriched_count = sum(1 for c in company_universe.values() 
                            if c.get('market_data', {}).get('realtime', {}).get('success', False))
        print(f"\n📊 {enriched_count} entreprises enrichies avec succès")
        
        # Ré-sauvegarder avec données enrichies
        print("\n💾 Sauvegarde du Company Universe enrichi...")
        with open(output_file_json, 'w', encoding='utf-8') as f:
            json.dump(company_universe, f, indent=2, ensure_ascii=False)
        print(f"✅ Fichier mis à jour: {output_file_json}")
        
        # Afficher un exemple d'enrichissement
        example_ticker = None
        for ticker, data in company_universe.items():
            if data.get('market_data', {}).get('realtime', {}).get('success', False):
                example_ticker = ticker
                break
        
        if example_ticker:
            example = company_universe[example_ticker]['market_data']['realtime']
            print(f"\n📋 Exemple d'enrichissement ({example_ticker}):")
            print(f"   Prix CSV: ${example.get('csv_price', 0):.2f}")
            print(f"   Prix actuel: ${example.get('current_price', 0):.2f} ({example.get('price_change_pct', 0):+.2f}%)")
            print(f"   Beta: {example.get('beta', 1.0):.2f}")
            print(f"   Facteur volatilité: {example.get('volatility_factor', 1.0):.3f}")
        
    else:
        print("⚠️ Enrichissement désactivé (ENRICH_TOP_N = None)")
        print("💡 Pour activer: mettre ENRICH_TOP_N = 50 (par exemple)")
        
except ImportError as e:
    print(f"⚠️ Import error: {e}")
    print("💡 Installer yfinance: pip install yfinance")
    print("⚠️ Enrichissement ignoré, continuant avec données CSV seulement")
except Exception as e:
    print(f"⚠️ Erreur lors de l'enrichissement: {e}")
    print("⚠️ Continuant avec données CSV seulement")


## Étape 8 : Visualiser les Données Disponibles (Optionnel)


In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Configuration
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (12, 8)
    
    # Créer un graphique de disponibilité des données
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Disponibilité par source
    ax1 = axes[0, 0]
    source_counts = [
        stats['with_market_data'],
        stats['with_data_points'],
        stats['with_legal_data']
    ]
    source_labels = ['Market Data', 'Data Points 10-K', 'Legal Data']
    ax1.bar(source_labels, source_counts, color=['#2ecc71', '#3498db', '#e74c3c'])
    ax1.set_title('Disponibilité des Sources de Données', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Nombre d\'entreprises')
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. Combinaisons de sources
    ax2 = axes[0, 1]
    combinations = []
    for ticker, company in company_universe.items():
        src = company['data_sources']
        combo = []
        if src['has_market_data']:
            combo.append('M')
        if src['has_data_points']:
            combo.append('D')
        if src['has_legal_data']:
            combo.append('L')
        combinations.append(''.join(combo) if combo else 'None')
    
    from collections import Counter
    combo_counts = Counter(combinations)
    if combo_counts:
        ax2.barh(list(combo_counts.keys()), list(combo_counts.values()), color='#9b59b6')
        ax2.set_title('Combinaisons de Sources', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Nombre d\'entreprises')
        ax2.set_ylabel('Combinaison (M=Market, D=Data Points, L=Legal)')
    
    # 3. Distribution Market Cap (si disponible)
    ax3 = axes[1, 0]
    market_caps = [
        company.get('market_data', {}).get('market_cap')
        for company in company_universe.values()
        if company.get('market_data', {}).get('market_cap')
    ]
    if market_caps:
        ax3.hist([mc / 1e12 for mc in market_caps if mc], bins=30, color='#3498db', edgecolor='black')
        ax3.set_title('Distribution Market Cap (en billions USD)', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Market Cap (Billions USD)')
        ax3.set_ylabel('Nombre d\'entreprises')
        ax3.grid(axis='y', alpha=0.3)
    
    # 4. Présence géographique (EU vs NA)
    ax4 = axes[1, 1]
    eu_count = sum(
        1 for company in company_universe.values()
        if company.get('data_points', {}).get('geography', {}).get('has_eu', False)
    )
    na_count = sum(
        1 for company in company_universe.values()
        if company.get('data_points', {}).get('geography', {}).get('has_na', False)
    )
    geo_labels = ['Présence EU', 'Présence NA']
    geo_counts = [eu_count, na_count]
    ax4.bar(geo_labels, geo_counts, color=['#16a085', '#27ae60'])
    ax4.set_title('Présence Géographique (depuis 10-K)', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Nombre d\'entreprises')
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'company_universe_visualization.png', dpi=150, bbox_inches='tight')
    print(f"✅ Visualisation sauvegardée: {OUTPUT_DIR / 'company_universe_visualization.png'}")
    plt.show()
except ImportError:
    print("⚠️ matplotlib/seaborn non disponibles - visualisation ignorée")


## Étape 9 : Exemple d'Entreprise Complète


In [ ]:
# Afficher un exemple d'entreprise avec le plus de données disponibles
example_companies = []
for ticker, company in company_universe.items():
    src = company['data_sources']
    score = sum([src['has_market_data'], src['has_data_points'], src['has_legal_data']])
    if score >= 2:  # Au moins 2 sources
        example_companies.append((ticker, company, score))

# Trier par score décroissant
example_companies.sort(key=lambda x: x[2], reverse=True)

if example_companies:
    best_ticker, best_company, score = example_companies[0]
    print(f"📊 Exemple d'entreprise avec données complètes: {best_ticker}")
    print(f"   Score de complétude: {score}/3")
    print(f"\n📋 Structure complète (premiers 2000 caractères):")
    print(json.dumps(best_company, indent=2, ensure_ascii=False)[:2000] + "...")
else:
    print("⚠️ Aucune entreprise avec au moins 2 sources de données")


## 📝 Notes et Prochaines Étapes

### ✅ Ce qui a été fait :
- Fusion Key Market Data + Data Points 10-K
- Structure prête pour Legal Data
- Statistiques et visualisations

### 🔄 À faire :
- ✅ Intégrer Legal Data quand disponible
- ✅ Enrichir avec APIs externes (Yahoo Finance, SEC, Morningstar)
- ✅ Créer index de recherche (pour RAG)
- ✅ Optimiser pour analyse d'impact réglementaire
